In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Rauschmanagement mit Estimator konfigurieren

<Accordion>
<AccordionItem title="Paketversionen">

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen, diese oder neuere Versionen zu verwenden.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Es gibt verschiedene Möglichkeiten, Rauschen zu managen – typischerweise durch den Einsatz verschiedener Fehlerminderungsund Fehlerunterdrückungstechniken, um Fehler zu vermeiden, bevor sie auftreten. Diese Techniken verursachen in der Regel Vorverarbeitungs-Overhead. Daher ist es wichtig, eine Balance zwischen der Verbesserung deiner Ergebnisse und der Sicherstellung zu finden, dass dein Job in angemessener Zeit abgeschlossen wird.

Estimator unterstützt die folgenden Rauschmanagement-Techniken. Eine Erklärung jeder Technik findest du unter [Fehlerminderungs- und Unterdrückungstechniken](error-mitigation-and-suppression-techniques). Anweisungen zur Aktivierung dieser Techniken findest du im Abschnitt [Benutzerdefinierte Fehlereinstellungen](#advanced-error).

- [dynamical decoupling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options#dynamicaldecouplingoptions)
- [Pauli twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
- [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
- [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
- [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
- [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)
- [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)

<span id="resilience"></span>
## Resilience-Level
Der `resilience_level` gibt an, wie viel Robustheit gegenüber
Fehlern aufgebaut werden soll. Höhere Level erzeugen genauere Ergebnisse, auf Kosten
längerer Verarbeitungszeiten. Resilience-Level können verwendet werden, um den
Kosten-/Genauigkeits-Kompromiss beim Anwenden von Rauschmanagement auf deine Primitive-
Abfrage zu konfigurieren. Rauschmanagement reduziert Fehler (Bias) in den Ergebnissen, indem die
Ausgaben einer Sammlung oder eines Ensembles verwandter Circuits verarbeitet werden. Der
Grad der Fehlerreduzierung hängt von der angewendeten Methode ab. Der Resilience-
Level abstrahiert die detaillierte Wahl der Rauschmanagement-Methode, damit
Nutzer den für ihre Anwendung geeigneten Kosten-/Genauigkeits-Kompromiss abwägen können.

Angesichts dessen entspricht jedes Level einer oder mehreren Methoden mit
zunehmendem Quantum-Sampling-Overhead, damit du verschiedene Zeit-/Genauigkeits-Kompromisse
ausprobieren kannst. Die folgende Tabelle zeigt, welche Level und die dazu gehörigen
Methoden für jedes der Primitives verfügbar sind.

<span id="resilience-table"></span>

| Resilience-Level | Beschreibung                                                                                            | Technik                                                                 |
|------------------|--------------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------|
| 0                | Keine Minderung                                                                                        | Keine                                                                   |
| 1 [Standard]     | Minimale Minderungskosten: Minderung von Fehlern durch Auslesefehler                                  | [Twirled Readout Error eXtinction (TREX)](/guides/error-mitigation-and-suppression-techniques#trex) Messung-Twirling |
| 2                | Mittlere Minderungskosten. Reduziert typischerweise Bias in Estimatoren, ist aber nicht garantiert zero-bias. | Level 1 + [Zero Noise Extrapolation (ZNE)](/guides/error-mitigation-and-suppression-techniques#zne) und Gate-Twirling

> **Info:** Resilience-Level befinden sich derzeit in der Beta-Phase, sodass Sampling-Overhead und
> Lösungsqualität je nach Circuit variieren. Neue Funktionen,
> erweiterte Optionen und Management-Tools werden fortlaufend
> veröffentlicht. Es ist nicht garantiert, dass auf jedem Resilience-Level bestimmte
> Rauschmanagement-Methoden angewendet werden.

### Estimator mit Resilience-Leveln konfigurieren
Du kannst Resilience-Level verwenden, um Rauschmanagement-Techniken festzulegen, oder du kannst benutzerdefinierte Techniken individuell einstellen, wie in [Benutzerdefinierte Fehlereinstellungen](#advanced-error) beschrieben.

> **Note:** Alle Optionen, die du zusätzlich zum Resilience-Level manuell angibst, werden zusätzlich zu den vom Resilience-Level definierten Basisoptionen angewendet. Daher könntest du prinzipiell den Resilience-Level auf 1 setzen und dann die Messfehlerminderung deaktivieren, obwohl dies nicht empfohlen wird.
> 
> Wenn du beispielsweise den Resilience-Level auf 0 setzt, wird `zne_mitigation` deaktiviert, aber `estimator.options.resilience.zne_mitigation = True` überschreibt diesen Wert.

### Beispiel
Der folgende Code aktiviert ZNE, TREX und Gate-Twirling durch
das Setzen von `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Benutzerdefinierte Rauschmanagement-Einstellungen
Du kannst einzelne Rauschmanagement-Methoden aktivieren und deaktivieren, indem du die [Estimator-Optionen](/guides/estimator-options) verwendest.

> **Note:** Nicht alle Optionen funktionieren zusammen mit allen Arten von Circuits. Weitere Details findest du in der [Funktionskompatibilitätstabelle](estimator-options#options-compatibility-table).

### Beispiel

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: "
    f"{estimator.options.twirling.enable_gates}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>